# AI Cycling Coach — GPU Training (Colab)

**Runtime → Change runtime type → T4 GPU** before running.

## Just click "Run all" — no uploads needed

The notebook generates synthetic training data directly here in Colab (~3 min for 20K athletes on CPU), then immediately trains on the GPU.

Steps:
1. Check GPU
2. Clone repo + install deps
3. Generate 20K athletes (~3 min on Colab CPU) — or change `ATHLETES = 50_000` for a fuller dataset (~8 min)
4. Train on GPU (~1–2 h on T4)
5. Save model to Drive + push to GitHub

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO = 'https://github.com/yossibello/ai-coach.git'

if not os.path.exists('/content/ai-coach'):
    !git clone {REPO} /content/ai-coach
else:
    !cd /content/ai-coach && git pull

%cd /content/ai-coach

import sys
sys.path.insert(0, '/content/ai-coach/backend')
os.environ['PYTHONPATH'] = '/content/ai-coach/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
!pip install pandas pyarrow -q
# torch is pre-installed on Colab with CUDA support
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ── 4. Load / generate training data ─────────────────────────────────────────
# THREE OPTIONS — set the MODE variable below:
#
#   'generate'  → generate data here in Colab (no upload needed, ~3 min)
#   'upload'    → you manually uploaded synthetic.parquet via the sidebar
#   'drive'     → file is on Google Drive at DRIVE_PATH

import os, shutil, pandas as pd

MODE       = 'generate'   # ← 'generate' | 'upload' | 'drive'
ATHLETES   = 20_000       # only used when MODE='generate'  (20K ≈ 3 min, 50K ≈ 8 min)
DRIVE_PATH = '/content/drive/MyDrive/ai-coach-data/synthetic.parquet'
DATA_FILE  = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if os.path.exists(DATA_FILE):
    print('✓ Data already in place, skipping.')

elif MODE == 'generate':
    import multiprocessing
    workers = max(1, multiprocessing.cpu_count() - 1)
    print(f'Generating {ATHLETES:,} athletes using {workers} workers…')
    !python -m ml.training.generate_synthetic \
        --athletes {ATHLETES} \
        --workers  {workers} \
        --output   {DATA_FILE}

elif MODE == 'upload':
    UPLOAD_PATH = '/content/synthetic.parquet'
    if not os.path.exists(UPLOAD_PATH):
        raise FileNotFoundError('Upload synthetic.parquet via sidebar → Files → Upload first')
    print(f'Copying uploaded file ({os.path.getsize(UPLOAD_PATH)/1e6:.0f} MB)…')
    shutil.copy(UPLOAD_PATH, DATA_FILE)

elif MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Copying from Drive ({os.path.getsize(DRIVE_PATH)/1e6:.0f} MB)…')
    shutil.copy(DRIVE_PATH, DATA_FILE)

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
print(f'✓ Data ready: {len(df):,} rows, {df.athlete_id.nunique():,} athletes, {len(df.columns)} cols')

In [ ]:

# ── 5. Train ──────────────────────────────────────────────────────────────────
import torch, os, sys, subprocess

# Auto-tune batch size based on available VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram_gb >= 38:
    BATCH_SIZE = 1024   # A100 80 GB / H100
elif vram_gb >= 20:
    BATCH_SIZE = 512    # A100 40 GB / RTX 3090+
else:
    BATCH_SIZE = 256    # T4 16 GB

EPOCHS     = 100
MODEL_FILE = 'backend/models/cycling_coach.pt'

print(f'GPU VRAM: {vram_gb:.1f} GB → batch size: {BATCH_SIZE}')

# Sanity checks before launching subprocess
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Data file not found: {DATA_FILE}\n'
        'Re-run the data cell (cell 4) first!'
    )
print(f'Data file OK: {os.path.getsize(DATA_FILE)/1e6:.0f} MB')

# Build env — make sure backend is importable from the subprocess
env = os.environ.copy()
backend_path = '/content/ai-coach/backend'
existing = env.get('PYTHONPATH', '')
env['PYTHONPATH'] = backend_path + (':' + existing if existing else '')

cmd = [
    sys.executable, '-m', 'ml.training.train',
    '--data',        DATA_FILE,
    '--output',      MODEL_FILE,
    '--epochs',      str(EPOCHS),
    '--batch-size',  str(BATCH_SIZE),
    '--patience',    '20',
]
print('Command:', ' '.join(cmd))
print('PYTHONPATH:', env['PYTHONPATH'])
print('─' * 60)

# Stream output to cell AND to training.log
with open('training.log', 'w', buffering=1) as log:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        cwd='/content/ai-coach',
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    proc.wait()

if proc.returncode != 0:
    print(f'\n❌ Training failed (exit code {proc.returncode})')
    raise RuntimeError('Training failed — see output above for details.')
else:
    print(f'\n✓ Training complete! Model saved to {MODEL_FILE}')


In [ ]:
# ── 6a. Save model to Google Drive (survives session end) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/ai-coach-models/'
os.makedirs(dst, exist_ok=True)
shutil.copy('backend/models/cycling_coach.pt', dst)
print('Saved to Google Drive:', dst + 'cycling_coach.pt')

In [ ]:
# ── 6b. Push model back to GitHub ─────────────────────────────────────────────
# You need a GitHub Personal Access Token (PAT) with repo write access.
# Create one at: https://github.com/settings/tokens  (Classic, repo scope)

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'colab@training'
!git config user.name 'Colab Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Colab GPU)"
!git push origin main
print('Model pushed to GitHub!')

In [ ]:
# ── 7. Quick sanity check ─────────────────────────────────────────────────────
import torch, sys
sys.path.insert(0, '/content/ai-coach/backend')
from app.ml.model import CyclingTransformer

m = CyclingTransformer()
m.load_state_dict(torch.load('backend/models/cycling_coach.pt', map_location='cpu'))
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()), )

# Show last training metrics
!tail -10 training.log